In [1]:
import os
os.chdir("..")

import sys
sys.path.append("src")

import pandas as pd

from wildfire_gnn.pipelines.cnn_baseline_pipeline import CNNBaselinePipeline
from wildfire_gnn.utils.config import load_yaml_config

In [2]:
config = load_yaml_config("configs/cnn_baseline_config.yaml")
pipeline = CNNBaselinePipeline(config)

In [3]:
from pathlib import Path
import rasterio

paths = list(Path("data/interim/aligned").glob("*.img"))

for p in paths:
    with rasterio.open(p) as src:
        print(p.name, src.shape, src.dtypes, src.nodata)

Burn_Prob.img (7597, 7555) ('float32',) -9999.0
CFL.img (7597, 7555) ('float32',) -9999.0
FSP_Index.img (7597, 7555) ('float32',) -9999.0
Fuel_Models.img (7597, 7555) ('uint8',) 0.0
Ignition_Prob.img (7597, 7555) ('float32',) -9999.0
Struct_Exp_Index.img (7597, 7555) ('float32',) -9999.0


In [4]:
results_df = pipeline.run()
results_df

2026-04-02 18:32:57 | INFO | cnn_baseline_pipeline | Stratified bin [0.0, 0.01): available=5356465 sampled=40000
2026-04-02 18:32:58 | INFO | cnn_baseline_pipeline | Stratified bin [0.01, 0.05): available=4727847 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.05, 0.1): available=1160219 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.1, 0.25): available=507855 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.25, 1.0): available=16 sampled=16
2026-04-02 18:33:04 | INFO | cnn_baseline_pipeline | Applied stratified subsampling: kept 200000 / 11752402 patches
2026-04-02 18:33:06 | INFO | cnn_baseline_pipeline | Saved patch metadata to data/processed/cnn_patch_metadata.csv
2026-04-02 18:33:06 | INFO | cnn_baseline_pipeline | CNN dataset stats: original_samples=11752402 sampled_samples=200000 target_min=0.000000 target_max=0.250882 target_mean=0.052204 target_std=0.053710
2026-04-02

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,cnn,random,val,0.022576,0.015515,0.819484,0.912219,0.931534
1,cnn,random,test,0.022593,0.015526,0.821805,0.914212,0.932958
2,cnn,spatial,val,0.025600,0.020139,0.705151,0.846233,0.874101
3,cnn,spatial,test,0.018484,0.015531,0.487520,0.848473,0.800038


In [5]:
metrics_df = pd.read_csv("reports/tables/cnn_metrics.csv")
metrics_df

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,cnn,random,val,0.022576,0.015515,0.819484,0.912219,0.931534
1,cnn,random,test,0.022593,0.015526,0.821805,0.914212,0.932958
2,cnn,spatial,val,0.025600,0.020139,0.705151,0.846233,0.874101
3,cnn,spatial,test,0.018484,0.015531,0.487520,0.848473,0.800038


In [6]:
cnn_random_preds = pd.read_csv("reports/tables/cnn_random_test_predictions.csv")
cnn_random_preds.head()

,row_index,col_index,y_true,y_pred,residual
0,4756,2197,0.088232,0.085811,0.002421
1,3728,2555,0.175064,0.089820,0.085244
2,1527,1837,0.122934,0.092039,0.030895
3,4296,2558,0.064533,0.056204,0.008329
4,1566,2981,0.047159,0.078844,-0.031685


In [7]:
cnn_spatial_preds = pd.read_csv("reports/tables/cnn_spatial_test_predictions.csv")
cnn_spatial_preds.head()

,row_index,col_index,y_true,y_pred,residual
0,4890,1812,0.019513,0.030981,-0.011469
1,1587,2502,0.033439,0.056645,-0.023205
2,846,3546,0.003566,0.031047,-0.027481
3,2531,1205,0.005752,0.022612,-0.016861
4,2201,1277,0.000667,0.017680,-0.017013


# 🔥 Phase 4B — CNN Baseline for Wildfire Burn Probability Prediction

##  Objective

The objective of Phase 4B is to build a **deep learning baseline** using Convolutional Neural Networks (CNNs) to predict wildfire burn probability from spatial raster data.

This phase extends the previous tabular baselines (Phase 4A) by incorporating:

- Local spatial context
- Non-linear feature interactions
- Patch-based representation of geospatial data

The CNN serves as a **strong intermediate benchmark** before moving to graph-based modeling in Phase 5.

---

##  Methodology Overview

###  Input Data

We used the following aligned raster layers:

**Target**
- Burn_Prob.img (burn probability)

**Features**
- CFL.img
- FSP_Index.img
- Ignition_Prob.img
- Struct_Exp_Index.img
- Fuel_Models.img

All rasters were:

- Spatially aligned to the same grid
- Cleaned from nodata values
- Standardized (for continuous features)

---

###  Patch-Based Learning

Instead of modeling individual pixels, we extracted **spatial patches**:

- Patch size: 15 × 15
- Input shape: [channels, height, width] = [5, 15, 15]

Each sample represents a **local neighborhood**, allowing the model to learn:

- Fire spread patterns
- Spatial dependencies
- Local environmental interactions

---

##  Dataset Strategy

###  Initial Challenge

The dataset contains approximately **11.7 million samples**, which creates:

- Heavy computational cost
- Severe imbalance (dominance of near-zero burn probability)

---

###  Solution: Stratified Subsampling

We applied:

- max_patch_samples = 200,000
- Stratified sampling based on burn probability bins

Bins:

- [0.0, 0.01)
- [0.01, 0.05)
- [0.05, 0.1)
- [0.1, 0.25)

This ensures:

- Balanced representation of low, medium, and high fire risk
- Reduced bias toward near-zero targets
- Scientifically meaningful sampling for publication

---

##  Training Setup

| Parameter | Value |
|----------|------|
| Batch size | 256 |
| Max epochs | 15 |
| Early stopping | 4 |
| Optimizer | AdamW |
| Loss function | MSE |
| Device | CPU |

---

##  Evaluation Strategy

We evaluated the model using two splitting strategies:

### 1. Random Split
- Random train/val/test division
- Measures interpolation performance

### 2. Spatial Split
- Geographic partitioning
- Measures generalization to unseen regions
- More realistic for real-world wildfire prediction

---

##  Results Summary

| Split | RMSE | MAE | R² | Pearson | Spearman |
|------|------|------|------|------|------|
| Random (Test) | 0.0226 | 0.0155 | 0.8218 | 0.914 | 0.933 |
| Spatial (Test) | 0.0185 | 0.0155 | 0.4875 | 0.848 | 0.800 |

---

##  Key Observations

###  Strong Performance on Random Split

- High R² (~0.82)
- Strong correlation (>0.9)

This indicates:

- The model successfully learns relationships within the observed data
- CNN captures local spatial dependencies effectively

---

###  Performance Drop in Spatial Split

- R² drops significantly (~0.49)
- Correlation decreases

This shows:

- Difficulty in generalizing to unseen geographic regions
- Presence of spatial distribution shift

This is expected in geospatial modeling and is a **critical research insight**.

---

##  Loss Curve Analysis

### Random Split

- Smooth convergence
- Stable validation loss for most epochs
- Slight overfitting observed in later epochs

### Spatial Split

- Faster convergence initially
- Early stopping triggered
- More unstable validation behavior

Conclusion:

- Model trains effectively
- Spatial generalization remains harder

---

##  Prediction Analysis

### Random Split

- Predictions closely follow the diagonal (ideal line)
- Strong alignment between predicted and true values
- Slight underestimation for higher burn probabilities

### Spatial Split

- Greater scatter in predictions
- Reduced accuracy in high-risk areas

Insight:

- Model captures general trends
- Struggles with regional variability

---

## Target Distribution

The burn probability distribution is:

- Highly skewed toward low values
- Long-tailed toward high-risk values

Implications:

- Justifies stratified sampling
- Explains model difficulty in predicting rare high-risk events

---

##  What We Achieved

###  Technical Achievements

- Built a full CNN pipeline from scratch
- Implemented:
  - Patch extraction
  - Stratified sampling
  - Random and spatial splits
- Generated:
  - Metrics
  - Predictions
  - Visualizations

---

###  Scientific Insights

1. CNN significantly improves over tabular baselines
2. Spatial generalization is a major challenge
3. Burn probability is highly imbalanced
4. Local spatial context is useful but not sufficient

---

###  Research Positioning

This phase establishes:

"A strong deep learning baseline that highlights the limitations of local spatial modeling."

---

##  Why This Matters for Your Thesis

CNN results clearly show:

- Good interpolation ability
- Weak extrapolation to new regions

This directly motivates the need for:

- Graph-based modeling
- Global spatial reasoning
- Connectivity-aware learning



##  Final Conclusion

Phase 4B successfully delivers:

- A scalable CNN baseline
- Strong predictive performance
- Critical insights about spatial limitations

Most importantly:

It proves that wildfire modeling requires **beyond-local spatial reasoning**, setting a strong foundation for GNN-based approaches.
